In [37]:
import pandas as pd
from textblob import TextBlob
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

LINUX_PATH= "~/S9/NLP/dataset/3.csv"
WIN_PATH= "~./dataset/3.csv"


nltk.download("punkt")
nltk.download("stopwords")
stop_words = set(stopwords.words("english"))


data = [
    ("Hey, are we still meeting for lunch at 1pm today?", "ham"),
    ("WINNER!! You have been selected to receive a £500 cash prize. Call 09061234567 now to claim!", "spam"),
    ("Can you pick up some milk on your way home?", "ham"),
    ("URGENT! Your mobile number has won a FREE holiday. Text CLAIM to 81010 to collect.", "spam"),
    ("I'll be late, the bus is stuck in traffic. Start without me.", "ham"),
    ("Congratulations! You've been chosen for a free iPhone. Click www.free-prize-now.com to claim.", "spam"),
    ("Thanks for helping me with my homework yesterday, I really appreciate it.", "ham"),
    ("FreeMsg: Get 6 months of unlimited calls for FREE! Reply YES to 87121. T&Cs apply.", "spam"),
    ("Don't forget mom's birthday is on Sunday, I'll bring the cake.", "ham"),
    ("Your bank account has been suspended. Verify your details now at www.secure-bank-login.net", "spam"),
    ("Did you finish the report? The boss is asking about it.", "ham"),
    ("You have 1 new voicemail. Call 08712400602 to listen. Cost 50p per min.", "spam"),
    ("Good night, see you tomorrow morning at the gym.", "ham"),
    ("Last chance! Win £1000 cash every week. Text WIN to 80082 now. 16+ only.", "spam"),
    ("Happy birthday! Hope you have an amazing day with your family.", "ham"),
]

df = pd.DataFrame(data, columns=["text", "label"])
df["label"] = df["label"].map({"ham": 0, "spam": 1})
y = df["label"]

print(df)

                                                 text  label
0   Hey, are we still meeting for lunch at 1pm today?      0
1   WINNER!! You have been selected to receive a £...      1
2         Can you pick up some milk on your way home?      0
3   URGENT! Your mobile number has won a FREE holi...      1
4   I'll be late, the bus is stuck in traffic. Sta...      0
5   Congratulations! You've been chosen for a free...      1
6   Thanks for helping me with my homework yesterd...      0
7   FreeMsg: Get 6 months of unlimited calls for F...      1
8   Don't forget mom's birthday is on Sunday, I'll...      0
9   Your bank account has been suspended. Verify y...      1
10  Did you finish the report? The boss is asking ...      0
11  You have 1 new voicemail. Call 08712400602 to ...      1
12   Good night, see you tomorrow morning at the gym.      0
13  Last chance! Win £1000 cash every week. Text W...      1
14  Happy birthday! Hope you have an amazing day w...      0


[nltk_data] Downloading package punkt to /home/ayman/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /home/ayman/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [38]:
def preprocess_text(text:str):
    tokens = word_tokenize(text)

    filt_word = [word for word in tokens if word not in stop_words and word.isalpha()]
    return " ".join(filt_word)

In [39]:
df["clean_text"] = df["text"].apply(preprocess_text)

In [40]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(df["clean_text"])

print(X.toarray())
df

[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 1 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 1 0 ... 0 0 0]]


,text,label,clean_text
0,"Hey, are we still meeting for lunch at 1pm today?",0,Hey still meeting lunch today
1,WINNER!! You have been selected to receive a £...,1,WINNER You selected receive cash prize Call claim
2,Can you pick up some milk on your way home?,0,Can pick milk way home
3,URGENT! Your mobile number has won a FREE holi...,1,URGENT Your mobile number FREE holiday Text CL...
4,"I'll be late, the bus is stuck in traffic. Sta...",0,I late bus stuck traffic Start without
5,Congratulations! You've been chosen for a free...,1,Congratulations You chosen free iPhone Click c...
6,Thanks for helping me with my homework yesterd...,0,Thanks helping homework yesterday I really app...
7,FreeMsg: Get 6 months of unlimited calls for F...,1,FreeMsg Get months unlimited calls FREE Reply ...
8,"Don't forget mom's birthday is on Sunday, I'll...",0,Do forget mom birthday Sunday I bring cake
9,Your bank account has been suspended. Verify y...,1,Your bank account suspended Verify details


In [44]:
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier

skf = StratifiedKFold(n_splits=5 , shuffle=True , random_state=42)

model = RandomForestClassifier()

history_fold = []

for fold, (train_index , val_index) in enumerate(skf.split(X,y)):
    X_train, X_val = X[train_index], X[val_index]
    y_train, y_val = y[train_index], y[val_index]

     # 4. Train the model on the training data for this fold
    model.fit(X_train, y_train)

    # 5. Evaluate on the validation data for this fold
    y_pred = model.predict(X_val)
    accuracy = sum(y_pred == y_val) / len(y_val) # Simple accuracy calculation
    history_fold.append(accuracy)
    print(f"  Fold {fold+1}: Validation Accuracy = {accuracy:.4f}")

# 6. Calculate average performance
mean_accuracy = sum(history_fold) / len(history_fold)
print(f"\nAverage Validation Accuracy: {mean_accuracy:.4f}")

  Fold 1: Validation Accuracy = 1.0000
  Fold 2: Validation Accuracy = 0.6667
  Fold 3: Validation Accuracy = 0.6667
  Fold 4: Validation Accuracy = 0.3333
  Fold 5: Validation Accuracy = 0.6667

Average Validation Accuracy: 0.6667
